# Lab02S02 — Análise e Visualização de Dados

Análise das métricas de qualidade (CBO, DIT, LCOM) dos 1.000 repositórios Java
mais populares do GitHub, correlacionando com métricas de processo.

**Questões de Pesquisa:**
- RQ01: Popularidade (estrelas) vs qualidade
- RQ02: Maturidade (idade em anos) vs qualidade
- RQ03: Atividade (releases) vs qualidade
- RQ04: Tamanho (LOC) vs qualidade

**Bônus:** Correlação de Spearman + Pearson + gráficos de correlação

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.alpha'] = 0.3

print("Bibliotecas carregadas com sucesso!")

## 2. Carregar dados

In [ ]:
df = pd.read_csv('data/metricas_ck_1000repos.csv')

print(f"Repositorios carregados: {len(df)}")
print(f"Colunas: {list(df.columns)}")
print()
df.head(10)

In [ ]:
# Estatisticas descritivas gerais
cols_interesse = ['estrelas', 'idade_anos', 'releases', 'loc_total',
                  'cbo_media', 'dit_media', 'lcom_media']
cols_disponiveis = [c for c in cols_interesse if c in df.columns]
df[cols_disponiveis].describe().round(2)

## 3. Funções auxiliares

In [ ]:
QUALITY_METRICS = ['cbo_media', 'dit_media', 'lcom_media']
QUALITY_LABELS = {'cbo_media': 'CBO (m\u00e9dia)', 'dit_media': 'DIT (m\u00e9dia)', 'lcom_media': 'LCOM (m\u00e9dia)'}

def correlation_table(df, process_col, process_label):
    """Calcula correlacao de Spearman e Pearson entre uma metrica de processo e as de qualidade."""
    results = []
    for qm in QUALITY_METRICS:
        if qm not in df.columns:
            continue
        valid = df[[process_col, qm]].dropna()
        if len(valid) < 10:
            continue
        
        sp_r, sp_p = stats.spearmanr(valid[process_col], valid[qm])
        pe_r, pe_p = stats.pearsonr(valid[process_col], valid[qm])
        
        results.append({
            'Metrica Processo': process_label,
            'Metrica Qualidade': QUALITY_LABELS.get(qm, qm),
            'Spearman r': round(sp_r, 4),
            'Spearman p-value': f'{sp_p:.2e}',
            'Spearman sig.': 'Sim' if sp_p < 0.05 else 'Nao',
            'Pearson r': round(pe_r, 4),
            'Pearson p-value': f'{pe_p:.2e}',
            'Pearson sig.': 'Sim' if pe_p < 0.05 else 'Nao',
        })
    return pd.DataFrame(results)


def scatter_plots(df, process_col, process_label, log_x=False):
    """Gera scatter plots de correlacao."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Correla\u00e7\u00e3o: {process_label} vs M\u00e9tricas de Qualidade', fontsize=14, fontweight='bold')
    
    for i, qm in enumerate(QUALITY_METRICS):
        if qm not in df.columns:
            continue
        ax = axes[i]
        valid = df[[process_col, qm]].dropna()
        
        ax.scatter(valid[process_col], valid[qm], alpha=0.3, s=15, color='steelblue')
        
        # Linha de tendencia
        if len(valid) > 2:
            z = np.polyfit(valid[process_col], valid[qm], 1)
            p = np.poly1d(z)
            x_sorted = np.sort(valid[process_col])
            ax.plot(x_sorted, p(x_sorted), 'r-', alpha=0.7, linewidth=2, label='Tendencia')
        
        sp_r, sp_p = stats.spearmanr(valid[process_col], valid[qm])
        ax.set_xlabel(process_label)
        ax.set_ylabel(QUALITY_LABELS.get(qm, qm))
        ax.set_title(f'{QUALITY_LABELS.get(qm, qm)}\n(Spearman r={sp_r:.3f}, p={sp_p:.2e})')
        if log_x and valid[process_col].min() > 0:
            ax.set_xscale('log')
        ax.legend()
    
    plt.tight_layout()
    plt.savefig(f'data/grafico_{process_col}.png', dpi=150, bbox_inches='tight')
    plt.show()


def summary_table(df, process_col, process_label, n_groups=4):
    """Divide repos em quartis pela metrica de processo e mostra medias de qualidade."""
    temp = df[[process_col] + [q for q in QUALITY_METRICS if q in df.columns]].dropna()
    temp['grupo'] = pd.qcut(temp[process_col], n_groups, labels=[f'Q{i+1}' for i in range(n_groups)], duplicates='drop')
    
    print(f'\n{process_label} - Medias por quartil:')
    summary = temp.groupby('grupo')[QUALITY_METRICS].agg(['mean', 'median', 'std']).round(4)
    display(summary)
    return summary

print('Funcoes auxiliares carregadas!')

---
## RQ01: Popularidade (estrelas) vs Qualidade

**Hipótese:** Repositórios mais populares tendem a ter melhor qualidade (menor CBO e LCOM, maior modularidade), pois recebem mais revisão de código e contribuições de desenvolvedores experientes.

In [ ]:
print('=== RQ01: Popularidade (Estrelas) vs Qualidade ===')
print()

corr_rq01 = correlation_table(df, 'estrelas', 'Estrelas')
display(corr_rq01)

scatter_plots(df, 'estrelas', 'Estrelas', log_x=True)
summary_table(df, 'estrelas', 'Estrelas')

---
## RQ02: Maturidade (idade em anos) vs Qualidade

**Hipótese:** Repositórios mais antigos tendem a acumular débito técnico, resultando em maior CBO e LCOM. Por outro lado, projetos maduros podem ter passado por refatorações que melhoram a qualidade.

In [ ]:
print('=== RQ02: Maturidade (Idade em Anos) vs Qualidade ===')
print()

corr_rq02 = correlation_table(df, 'idade_anos', 'Idade (anos)')
display(corr_rq02)

scatter_plots(df, 'idade_anos', 'Idade (anos)')
summary_table(df, 'idade_anos', 'Idade (anos)')

---
## RQ03: Atividade (releases) vs Qualidade

**Hipótese:** Repositórios com mais releases tendem a ter melhor qualidade, pois o ciclo frequente de releases indica um processo de desenvolvimento mais disciplinado, com CI/CD e revisão de código.

In [ ]:
print('=== RQ03: Atividade (Releases) vs Qualidade ===')
print()

corr_rq03 = correlation_table(df, 'releases', 'Releases')
display(corr_rq03)

scatter_plots(df, 'releases', 'N\u00ba de Releases', log_x=True)
summary_table(df, 'releases', 'Releases')

---
## RQ04: Tamanho (LOC) vs Qualidade

**Hipótese:** Repositórios maiores (mais LOC) tendem a ter pior qualidade (maior CBO e LCOM), pois projetos grandes são mais difíceis de manter modulares e coesos.

In [ ]:
print('=== RQ04: Tamanho (LOC total) vs Qualidade ===')
print()

if 'loc_total' in df.columns:
    corr_rq04 = correlation_table(df, 'loc_total', 'LOC total')
    display(corr_rq04)
    
    scatter_plots(df, 'loc_total', 'LOC total', log_x=True)
    summary_table(df, 'loc_total', 'LOC total')
else:
    print('Coluna loc_total nao disponivel.')

---
## 4. Visão Geral: Matriz de Correlação (Bônus)

In [ ]:
# Matriz de correlacao de Spearman completa
process_metrics = ['estrelas', 'idade_anos', 'releases']
if 'loc_total' in df.columns:
    process_metrics.append('loc_total')

all_metrics = process_metrics + [q for q in QUALITY_METRICS if q in df.columns]
corr_matrix = df[all_metrics].corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(len(all_metrics)))
ax.set_yticks(range(len(all_metrics)))
ax.set_xticklabels(all_metrics, rotation=45, ha='right')
ax.set_yticklabels(all_metrics)

# Anotar valores
for i in range(len(all_metrics)):
    for j in range(len(all_metrics)):
        val = corr_matrix.iloc[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=10)

plt.colorbar(im, label='Spearman r')
plt.title('Matriz de Correla\u00e7\u00e3o de Spearman\n(M\u00e9tricas de Processo vs Qualidade)', fontweight='bold')
plt.tight_layout()
plt.savefig('data/grafico_matriz_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Tabela Resumo de Todas as Correlações (Bônus)

In [ ]:
# Tabela consolidada de todas as correlacoes
all_corrs = []
pairs = [
    ('estrelas', 'Estrelas (Popularidade)'),
    ('idade_anos', 'Idade em anos (Maturidade)'),
    ('releases', 'N\u00ba Releases (Atividade)'),
]
if 'loc_total' in df.columns:
    pairs.append(('loc_total', 'LOC total (Tamanho)'))

for col, label in pairs:
    ct = correlation_table(df, col, label)
    all_corrs.append(ct)

df_all_corrs = pd.concat(all_corrs, ignore_index=True)
print('Tabela consolidada de correlacoes:')
display(df_all_corrs)

# Salvar
df_all_corrs.to_csv('data/tabela_correlacoes.csv', index=False)
print('Salvo: data/tabela_correlacoes.csv')

## 6. Distribuição das Métricas de Qualidade

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribui\u00e7\u00e3o das M\u00e9tricas de Qualidade (por reposit\u00f3rio)', fontsize=14, fontweight='bold')

for i, qm in enumerate(QUALITY_METRICS):
    if qm not in df.columns:
        continue
    ax = axes[i]
    data = df[qm].dropna()
    
    ax.hist(data, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(data.median(), color='red', linestyle='--', linewidth=2, label=f'Mediana={data.median():.2f}')
    ax.axvline(data.mean(), color='orange', linestyle='--', linewidth=2, label=f'M\u00e9dia={data.mean():.2f}')
    ax.set_xlabel(QUALITY_LABELS.get(qm, qm))
    ax.set_ylabel('Frequencia')
    ax.set_title(QUALITY_LABELS.get(qm, qm))
    ax.legend()

plt.tight_layout()
plt.savefig('data/grafico_distribuicoes.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Box Plots por Quartis (Bônus)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('M\u00e9tricas de Qualidade por Quartil das M\u00e9tricas de Processo', fontsize=14, fontweight='bold')

process_cols = [('estrelas', 'Estrelas'), ('idade_anos', 'Idade (anos)'),
                ('releases', 'Releases')]
if 'loc_total' in df.columns:
    process_cols.append(('loc_total', 'LOC total'))

for idx, (pcol, plabel) in enumerate(process_cols):
    ax = axes[idx // 2][idx % 2]
    
    temp = df[[pcol, 'cbo_media']].dropna()
    temp['quartil'] = pd.qcut(temp[pcol], 4, labels=['Q1\n(baixo)', 'Q2', 'Q3', 'Q4\n(alto)'], duplicates='drop')
    
    groups = [group['cbo_media'].values for name, group in temp.groupby('quartil', observed=True)]
    labels = [name for name, _ in temp.groupby('quartil', observed=True)]
    
    bp = ax.boxplot(groups, labels=labels, patch_artist=True)
    colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
    for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    ax.set_xlabel(plabel)
    ax.set_ylabel('CBO (m\u00e9dia)')
    ax.set_title(f'CBO por quartil de {plabel}')

plt.tight_layout()
plt.savefig('data/grafico_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Salvar dados para o relatório

In [ ]:
# Estatisticas descritivas para o relatorio
desc = df[cols_disponiveis].describe().round(4)
desc.to_csv('data/estatisticas_descritivas.csv')
print('Salvo: data/estatisticas_descritivas.csv')

print(f'\nTotal de graficos gerados na pasta data/:')
import glob
for f in sorted(glob.glob('data/grafico_*.png')):
    print(f'  {f}')

print(f'\nTotal de CSVs gerados:')
for f in sorted(glob.glob('data/*.csv')):
    print(f'  {f}')

print('\nAnalise concluida! Dados prontos para o relatorio.')